In [0]:
# Importing necessary libraries
from delta.tables import *
from pyspark.sql import functions as F



In [0]:
# Func to upload and upsert
def upsert_to_schema(df,target_schema,target_table,join_key):
    full_table_path=f"{target_schema}.{target_table}"
    
    if not spark.catalog.tableExists(full_table_path):
        print("Table does not exist, creating.....")
        df.write.format("delta").saveAsTable(full_table_path)
    else:
        print("Table exists, performing merge.....")
        target_delta_table= DeltaTable.forName(spark,full_table_path)

        target_delta_table.alias("target").
        merge(
            df.alias("source"),
            "target."+join_key+" = source."+join_key
        )
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()